# Relatório Base 2

Gera/atualiza `outputs/relatorio_base2.html` a partir dos artefatos já existentes.

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown
import pandas as pd

# sobe até achar analyses/grupo2
here = Path.cwd().resolve()
group = None
for c in [here, *here.parents]:
    if (c / 'config.py').exists() and (c / 'lib').is_dir():
        group = c
        break
if group is None:
    raise RuntimeError('Abra/rode este notebook de dentro de analyses/grupo2/...')
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib
from lib import features, eda_stl, models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print('grupo2 →', group)
print('outputs →', config.OUTPUT_DIR)


In [ ]:
from lib.report import write_report
from lib.models import ForecastResult

clean_report = load_json(require_file(config.DATA_DIR / 'cleaning_report.json', '02_limpeza'))
stl_info = load_json(require_file(config.RESULT_DIR / 'stl_summary.json', '05_STL'))
params = load_json(require_file(config.RESULT_DIR / 'best_params.json', 'tuning dos modelos'))
mae_df = pd.read_csv(require_file(config.RESULT_DIR / 'mae_consolidado.csv', '01_MAE'))
results = {}
for model in mae_df['model']:
    pred = pd.read_csv(config.RESULT_DIR / f'predictions_{model}.csv')
    meta = load_json(config.RESULT_DIR / f'metrics_{model}.json')
    fi_path = config.RESULT_DIR / f'feature_importance_{model}.csv'
    fi = pd.read_csv(fi_path) if fi_path.exists() else None
    results[model] = ForecastResult(model, meta['params'], pred, meta['mae'],
                                    meta['runtime_sec'], meta['ljung_box_pvalue_lag24'], fi)
path = write_report(clean_report, stl_info, params, mae_df, results)
print('HTML:', path)
display(mae_df)
